# Import des bibliothèques nécessaires

In [1]:
from pathlib import Path
import pandas as pd
import re



# Chargement des données

In [5]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "01_paquets_phrases.csv", encoding="utf-8",)
df.head()

,nom_fichier,id_paquet,phrases_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Elle s’était redressée, le sang aux joues, les..."
2,1893_20_Le_docteur_Pascal._clean.txt,2,"De face, dans son visage séché, ses yeux garda..."
3,1893_20_Le_docteur_Pascal._clean.txt,3,"pourquoi Pascal, qui aurait pu marcher sur leu..."
4,1893_20_Le_docteur_Pascal._clean.txt,4,"Certes, il n’y a pas de honte à cela; seulemen..."


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3556 entries, 0 to 3555
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nom_fichier     3556 non-null   str  
 1   id_paquet       3556 non-null   int64
 2   phrases_paquet  3556 non-null   str  
dtypes: int64(1), str(2)
memory usage: 83.5 KB


# Préparation du dataframe 

In [7]:
df = df.rename(columns={
    "nom_fichier": "fichier",
    "id_paquet": "paquet_id",
    "phrases_paquet": "texte"
})

In [8]:
df["annee"] = df["fichier"].str.extract(r"^(\d{4})").astype(int)

In [9]:
ordre_romans= (
    df[["fichier", "annee"]]
    .drop_duplicates()
    .sort_values(["annee", "fichier"])
    .reset_index(drop=True)
)

ordre_romans["ordre_romans"] = range(1, len(ordre_romans) + 1)

df = df.merge(ordre_romans[["fichier", "ordre_romans"]], on="fichier", how="left")

In [10]:
df["roman"] = (
    df["fichier"]
    .str.replace(r"^\d{4}_\d+_", "", regex=True)
    .str.replace(r"_clean\.txt$", "", regex=True)
    .str.replace("_", " ")
)

In [11]:
df["nb_mots"] = df["texte"].str.split().str.len()

In [12]:
df = df[[
    "roman",
    "annee",
    "ordre_romans",
    "paquet_id",
    "texte",
    "nb_mots"
]]

In [13]:
df = df.sort_values(["ordre_romans", "paquet_id"]).reset_index(drop=True)

In [14]:
df["paquet_id"] = df.groupby("roman").cumcount() + 1

In [15]:
df.head()

,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",985
1,1865 La confession de Claude.,1865,1,2,"Ai-je eu trop de confiance en ma force, ma pla...",1067
2,1865 La confession de Claude.,1865,1,3,Hélas! il me faut cependant une ombre de réali...,894
3,1865 La confession de Claude.,1865,1,4,Elle dormait. J’ai entassé sur ses pieds tous ...,1158
4,1865 La confession de Claude.,1865,1,5,"Ainsi, jamais mon cœur ne pourra battre sans q...",1027


In [16]:
chemin_sortie = Path("..") /"data" /"2_processed" /"02_corpus_zola.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")